# 🤗 LeRobot Quickstart

Calibration → teleoperation → data collection → training → evaluation.

Install the required dependencies: `pip install -e .[notebook,dataset,training,viz,hardware]`.

**How to use:**
1. Edit the **Configuration** cell with your settings.
2. Run all cells (`Run All`).
3. Each section prints a ready-to-paste terminal command - copy it and run it.

Each setup is different, please refer to the [LeRobot documentation](https://huggingface.co/docs/lerobot/il_robots) for more details on each step and available options. <br>
Feel free to make this notebook your own and adapt it to your needs!

In [10]:
from huggingface_hub import snapshot_download

# Remplacez "LohanTS/mon-dataset-robot" par le nom exact
local_dir = "/content/LohanTS/mon-dataset-robot"

snapshot_download(
    repo_id="LohanTS/mon-dataset-robot",
    repo_type="model",
    local_dir=local_dir
)

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

'/content/LohanTS/mon-dataset-robot'

---
## Utils

In [11]:
def _cameras_arg(cameras: dict) -> str:
    if not cameras:
        return ""
    entries = [f"{n}: {{{', '.join(f'{k}: {v}' for k, v in cfg.items())}}}" for n, cfg in cameras.items()]
    return "{ " + ", ".join(entries) + " }"


def print_cmd(*parts: str) -> None:
    """Print a shell command with line continuations, skipping empty parts."""
    non_empty = [p for p in parts if p]
    print(" \\\n    ".join(non_empty))

---
## Configuration

Edit this cell, then **Run All** to generate all commands below.

In [12]:
# Robot (follower) - run `lerobot-find-port` to discover the port
ROBOT_TYPE = "so101_follower"
ROBOT_PORT = "/dev/ttyACM0"
ROBOT_ID = "my_follower_arm"

# Teleop (leader) - run `lerobot-find-port` to discover the port
TELEOP_TYPE = "so101_leader"
TELEOP_PORT = "/dev/ttyACM1"
TELEOP_ID = "my_leader_arm"

# Cameras - set to {} to disable
# Run `lerobot-find-cameras opencv` to list available cameras and their indices
CAMERAS = {
    "top": {"type": "opencv", "index_or_path": 2, "width": 640, "height": 480, "fps": 30},
    "wrist": {"type": "opencv", "index_or_path": 4, "width": 640, "height": 480, "fps": 30},
}

# Dataset
HF_USER = "LohanTS"  # `hf auth whoami` to find your username
DATASET_NAME = "/content/LohanTS/mon-dataset-robot"
TASK_DESCRIPTION = "pick and place the block"
NUM_EPISODES = 10

# Training
POLICY_TYPE = "act"  # act, diffusion, smolvla, ...
POLICY_DEVICE = "cuda"  # cuda / cpu / mps
TRAIN_STEPS = 10_000
SAVE_FREQ = 2_000
OUTPUT_DIR = f"outputs/train/{DATASET_NAME}"

# Inference - Hub repo ID or local checkpoint path
# e.g. set to f"{OUTPUT_DIR}/checkpoints/last" to use a local checkpoint
POLICY_PATH = f"{HF_USER}/{DATASET_NAME}_{POLICY_TYPE}"
LAST_CHECKPOINT_PATH = f"{OUTPUT_DIR}/checkpoints/last"

# Derived
DATASET_REPO_ID = f"{HF_USER}/{DATASET_NAME}"
DATASET_ROOT = f"data/{DATASET_NAME}"
POLICY_REPO_ID = f"{HF_USER}/{DATASET_NAME}_{POLICY_TYPE}"
EVAL_REPO_ID = f"{HF_USER}/eval_{DATASET_NAME}"
CAMERAS_ARG = _cameras_arg(CAMERAS)
CAMERAS_FLAG = f'--robot.cameras="{CAMERAS_ARG}"' if CAMERAS_ARG else ""

print(f"Robot  : {ROBOT_TYPE} @ {ROBOT_PORT}")
print(f"Teleop : {TELEOP_TYPE} @ {TELEOP_PORT}")
print(f"Cameras: {list(CAMERAS) or 'none'}")
print(f"Dataset: {DATASET_REPO_ID} ({NUM_EPISODES} episodes) saved to {DATASET_ROOT}")
print(f"Policy : {POLICY_TYPE} -> {POLICY_REPO_ID}")

Robot  : so101_follower @ /dev/ttyACM0
Teleop : so101_leader @ /dev/ttyACM1
Cameras: ['top', 'wrist']
Dataset: LohanTS//content/LohanTS/mon-dataset-robot (10 episodes) saved to data//content/LohanTS/mon-dataset-robot
Policy : act -> LohanTS//content/LohanTS/mon-dataset-robot_act


---
## 1. Calibration

Run once per arm before first use.

In [13]:
# Follower
print_cmd(
    "lerobot-calibrate",
    f"--robot.type={ROBOT_TYPE}",
    f"--robot.port={ROBOT_PORT}",
    f"--robot.id={ROBOT_ID}",
)

lerobot-calibrate \
    --robot.type=so101_follower \
    --robot.port=/dev/ttyACM0 \
    --robot.id=my_follower_arm


In [14]:
# Leader
print_cmd(
    "lerobot-calibrate",
    f"--teleop.type={TELEOP_TYPE}",
    f"--teleop.port={TELEOP_PORT}",
    f"--teleop.id={TELEOP_ID}",
)

lerobot-calibrate \
    --teleop.type=so101_leader \
    --teleop.port=/dev/ttyACM1 \
    --teleop.id=my_leader_arm


---
## 2. Teleoperation

See the [teleoperation docs](https://huggingface.co/docs/lerobot/il_robots#teleoperate) and the [cameras guide](https://huggingface.co/docs/lerobot/cameras) for more options.

In [15]:
print_cmd(
    "lerobot-teleoperate",
    f"--robot.type={ROBOT_TYPE}",
    f"--robot.port={ROBOT_PORT}",
    f"--robot.id={ROBOT_ID}",
    CAMERAS_FLAG,
    f"--teleop.type={TELEOP_TYPE}",
    f"--teleop.port={TELEOP_PORT}",
    f"--teleop.id={TELEOP_ID}",
    "--display_data=true",
)

lerobot-teleoperate \
    --robot.type=so101_follower \
    --robot.port=/dev/ttyACM0 \
    --robot.id=my_follower_arm \
    --robot.cameras="{ top: {type: opencv, index_or_path: 2, width: 640, height: 480, fps: 30}, wrist: {type: opencv, index_or_path: 4, width: 640, height: 480, fps: 30} }" \
    --teleop.type=so101_leader \
    --teleop.port=/dev/ttyACM1 \
    --teleop.id=my_leader_arm \
    --display_data=true


---
## 3. Record Dataset

See the [recording docs](https://huggingface.co/docs/lerobot/il_robots#record-a-dataset) for tips on gathering good data.

In [16]:
print_cmd(
    "lerobot-record",
    f"--robot.type={ROBOT_TYPE}",
    f"--robot.port={ROBOT_PORT}",
    f"--robot.id={ROBOT_ID}",
    CAMERAS_FLAG,
    f"--teleop.type={TELEOP_TYPE}",
    f"--teleop.port={TELEOP_PORT}",
    f"--teleop.id={TELEOP_ID}",
    f"--dataset.repo_id={DATASET_REPO_ID}",
    f"--dataset.num_episodes={NUM_EPISODES}",
    f'--dataset.single_task="{TASK_DESCRIPTION}"',
    "--dataset.streaming_encoding=true",
    "--display_data=true",
)

lerobot-record \
    --robot.type=so101_follower \
    --robot.port=/dev/ttyACM0 \
    --robot.id=my_follower_arm \
    --robot.cameras="{ top: {type: opencv, index_or_path: 2, width: 640, height: 480, fps: 30}, wrist: {type: opencv, index_or_path: 4, width: 640, height: 480, fps: 30} }" \
    --teleop.type=so101_leader \
    --teleop.port=/dev/ttyACM1 \
    --teleop.id=my_leader_arm \
    --dataset.repo_id=LohanTS//content/LohanTS/mon-dataset-robot \
    --dataset.num_episodes=10 \
    --dataset.single_task="pick and place the block" \
    --dataset.streaming_encoding=true \
    --display_data=true


In [17]:
# Resume a previously interrupted recording session
print_cmd(
    "lerobot-record",
    f"--robot.type={ROBOT_TYPE}",
    f"--robot.port={ROBOT_PORT}",
    f"--robot.id={ROBOT_ID}",
    CAMERAS_FLAG,
    f"--teleop.type={TELEOP_TYPE}",
    f"--teleop.port={TELEOP_PORT}",
    f"--teleop.id={TELEOP_ID}",
    f"--dataset.repo_id={DATASET_REPO_ID}",
    f"--dataset.root={DATASET_ROOT}",
    f"--dataset.num_episodes={NUM_EPISODES}",
    f'--dataset.single_task="{TASK_DESCRIPTION}"',
    "--dataset.streaming_encoding=true",
    "--display_data=true",
    "--resume=true",
)

lerobot-record \
    --robot.type=so101_follower \
    --robot.port=/dev/ttyACM0 \
    --robot.id=my_follower_arm \
    --robot.cameras="{ top: {type: opencv, index_or_path: 2, width: 640, height: 480, fps: 30}, wrist: {type: opencv, index_or_path: 4, width: 640, height: 480, fps: 30} }" \
    --teleop.type=so101_leader \
    --teleop.port=/dev/ttyACM1 \
    --teleop.id=my_leader_arm \
    --dataset.repo_id=LohanTS//content/LohanTS/mon-dataset-robot \
    --dataset.root=data//content/LohanTS/mon-dataset-robot \
    --dataset.num_episodes=10 \
    --dataset.single_task="pick and place the block" \
    --dataset.streaming_encoding=true \
    --display_data=true \
    --resume=true


---
## 4. Train Policy

See the [training docs](https://huggingface.co/docs/lerobot/il_robots#train-a-policy) for configuration options and tips.

In [18]:
print_cmd(
    "lerobot-train",
    f"--dataset.repo_id={DATASET_REPO_ID}",
    f"--policy.type={POLICY_TYPE}",
    f"--policy.device={POLICY_DEVICE}",
    f"--policy.repo_id={POLICY_REPO_ID}",
    f"--output_dir={OUTPUT_DIR}",
    f"--steps={TRAIN_STEPS}",
    f"--save_freq={SAVE_FREQ}",
)

lerobot-train \
    --dataset.repo_id=LohanTS//content/LohanTS/mon-dataset-robot \
    --policy.type=act \
    --policy.device=cuda \
    --policy.repo_id=LohanTS//content/LohanTS/mon-dataset-robot_act \
    --output_dir=outputs/train//content/LohanTS/mon-dataset-robot \
    --steps=10000 \
    --save_freq=2000


In [1]:
!lerobot-train \
  --dataset.repo_id="LohanTS/mon-dataset-robot" \
  --policy.type="act" \
  --policy.device="cuda" \
  --policy.repo_id="LohanTS/mon-dataset-robot_act" \
  --output_dir="outputs/train/LohanTS/mon-dataset-robot" \
  --steps=10000 \
  --save_freq=2000

INFO 2026-06-25 14:49:01 ot_train.py:222 {'batch_size': 8,
 'checkpoint_path': None,
 'cudnn_deterministic': False,
 'dataset': {'episodes': None,
             'eval_split': 0.0,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                                                                   1.2

In [9]:
!ls -R /content/LohanTS/mon-dataset-robot

/content/LohanTS/mon-dataset-robot:
data  meta  videos

/content/LohanTS/mon-dataset-robot/data:
chunk-000

/content/LohanTS/mon-dataset-robot/data/chunk-000:
file-000.parquet

/content/LohanTS/mon-dataset-robot/meta:
episodes  info.json  stats.json  tasks.parquet

/content/LohanTS/mon-dataset-robot/meta/episodes:
chunk-000

/content/LohanTS/mon-dataset-robot/meta/episodes/chunk-000:
file-000.parquet

/content/LohanTS/mon-dataset-robot/videos:
observation.images.top

/content/LohanTS/mon-dataset-robot/videos/observation.images.top:
chunk-000

/content/LohanTS/mon-dataset-robot/videos/observation.images.top/chunk-000:
file-000.mp4


In [ ]:
!lerobot-train \
  --dataset.repo_id="LohanTS/mon-dataset-robot" \
  --dataset.root="/content/LohanTS/mon-dataset-robot" \
  --policy.type="act" \
  --policy.device="cuda" \
  --policy.repo_id="LohanTS/mon-model-robot" \
  --output_dir="outputs/train/LohanTS/mon-dataset-robot" \
  --steps=10000 \
  --save_freq=2000

INFO 2026-06-25 14:59:18 ot_train.py:222 {'batch_size': 8,
 'checkpoint_path': None,
 'cudnn_deterministic': False,
 'dataset': {'episodes': None,
             'eval_split': 0.0,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                                                                   1.2

In [24]:
# 1. On désinstalle la version qui contient le bug
!pip uninstall -y lerobot

# 2. On installe la version directement depuis GitHub
!pip install git+https://github.com/huggingface/lerobot.git

# 3. (Optionnel mais recommandé) Redémarrez la session du Notebook
# Menu "Exécution" > "Redémarrer la session"

Found existing installation: lerobot 0.5.1
Uninstalling lerobot-0.5.1:
  Successfully uninstalled lerobot-0.5.1
  Cloning https://github.com/huggingface/lerobot.git to /tmp/pip-req-build-8n8hw4wu
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/lerobot.git /tmp/pip-req-build-8n8hw4wu
  Resolved https://github.com/huggingface/lerobot.git to commit 6a788fbdb02cabfae60f7408636945df0b1eafa0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for lerobot: filename=lerobot-0.5.2-py3-none-any.whl size=1587969 sha256=e11582b26ede857eccaefd013bd5b9e10d2ee182e7c8101655b01c366d945493
  Stored in directory: /tmp/pip-ephem-wheel-cache-6rpczs6i/wheels/f9/58/7e/567ed447b73477c058314dea9c27a22f337be69d74a01f5f81
Successfully built lerobot


In [19]:
# Resume a previously interrupted training session
print_cmd(
    "lerobot-train",
    f"--config_path={LAST_CHECKPOINT_PATH}/pretrained_model/train_config.json",
    "--resume=true",
)

lerobot-train \
    --config_path=outputs/train//content/LohanTS/mon-dataset-robot/checkpoints/last/pretrained_model/train_config.json \
    --resume=true


In [20]:
from huggingface_hub import HfApi

# Initialisation de l'API
api = HfApi()

# Chemin local de vos fichiers
local_folder = "/content/LohanTS/mon-dataset-robot"

# Upload vers votre dépôt Dataset
api.upload_folder(
    folder_path=local_folder,
    repo_id="LohanTS/mon-dataset-robot",
    repo_type="dataset",
    commit_message="Transfert des données vers un dépôt Dataset"
)

print("Upload terminé ! Vos fichiers sont maintenant dans votre Dataset.")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...hunk-000/file-000.parquet: 100%|##########| 43.7kB / 43.7kB            

  ...hunk-000/file-000.parquet: 100%|##########| 78.1kB / 78.1kB            

  ...op/chunk-000/file-000.mp4: 100%|##########| 6.02MB / 6.02MB            

  ...-robot/meta/tasks.parquet: 100%|##########| 1.99kB / 1.99kB            

Upload terminé ! Vos fichiers sont maintenant dans votre Dataset.


---
## 5. Inference

Uses `POLICY_PATH` from the Configuration cell (defaults to the Hub repo ID). You can also put there the `LAST_CHECKPOINT_PATH`.

See the [inference docs](https://huggingface.co/docs/lerobot/il_robots#run-inference-and-evaluate-your-policy) for details.

Recently ```lerobot-rollout``` was introduced, you can [read more about it here](https://huggingface.co/docs/lerobot/main/en/il_robots?eval=Base+mode+%28no+recording%29#run-inference-and-evaluate-your-policy).

In [ ]:
print_cmd(
    "lerobot-rollout",
    "--strategy.type=base",
    f"--policy.path={POLICY_PATH}",
    f"--robot.type={ROBOT_TYPE}",
    f"--robot.port={ROBOT_PORT}",
    CAMERAS_FLAG,
    f'--task="{TASK_DESCRIPTION}"',
    "--duration=60",
)

if you are using the V0.5.1 release you should use ```lerobot-record``` instead of rollout

In [ ]:
print_cmd(
    "lerobot-record",
    f"--policy.path={POLICY_PATH}",
    f"--robot.type={ROBOT_TYPE}",
    f"--robot.port={ROBOT_PORT}",
    f"--robot.id={ROBOT_ID}",
    CAMERAS_FLAG,
    f"--teleop.type={TELEOP_TYPE}",
    f"--teleop.port={TELEOP_PORT}",
    f"--teleop.id={TELEOP_ID}",
    f"--dataset.repo_id={EVAL_REPO_ID}",
    f"--dataset.num_episodes={NUM_EPISODES}",
    f'--dataset.single_task="{TASK_DESCRIPTION}"',
    "--dataset.streaming_encoding=true",
)